In [1]:
import json
from pathlib import Path
import pandas as pd
import re
import extract as ext

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()

In [3]:
with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/temas_ejercicio.json", "r") as file:
    temas_ejercicio = json.load(file)

with open(ROOT / "diccionarios_clasificacion_ejercicios/quimica/quimica_keywords.json") as file:
    quimica_keywords = json.load(file)

In [ ]:
def classify_exercise(exam_name: str, statement: str, keywords = quimica_keywords) -> tuple:
    # obtener el año del examen
    exam_year = int(exam_name.split("_")[0])
    exam_name = (" - ".join(exam_name.split("_")[1:])).capitalize()

    # seleccionar modelo de examen en función del año
    if exam_year < 2020:
        exam_model = "modelo_2010_2019"
        exercise_number_pattern = r"^\d{1}\.-"

    elif exam_year < 2024:
        exam_model = "modelo_2020_2023"
        exercise_number_pattern = r"^[ABC]\d{1}"
    else:
        exam_model = "modelo_2024+"
        exercise_number_pattern = r"^\d{1}?[AB]"

    # temas del ejercicio según número ejercicio y modelo

    # "modelo_2024+": {
    #     "tipo_numeración" : "1A, 1B, 2A, 2B, ..., 4B, 5",
    #     "1": ["configuracion_electronica", ...]
    #     "2": ["equilibrio_quimico", ...],

    exercise_topics_dict = temas_ejercicio[exam_model]

    # obtener el número del ejercicio
    exercise_number = re.match(exercise_number_pattern, statement)

    # si se encuentra el número de ejercicio
    if exercise_number:

        # extraer la clave del diccionario del modelo de examen
        if exam_year < 2020:  # 2010 - 2019 - key 1,2,3,...
            exercise_number_key = exercise_number[0][0]

            # las keys son: "1", "2,3,4", "5,6"
            # exercise_number_key necesita un ajuste
            # para que se encuentre la clave
            for key in exercise_topics_dict.keys():
                if exercise_number_key in key:
                    exercise_number_key = key
                    break
        
        elif 2020 <= exam_year <= 2023:  # 2020 - 2023 - key A, B, C
            exercise_number_key = exercise_number[0][0]

        else:  # 2024 + - key 1, 2, 3, 4A, 4B, 5
            exercise_number_key = exercise_number[0]
            if exercise_number[0] not in  ["4A", "4B"]:
                exercise_number_key = exercise_number[0][0]
            else:
                exercise_number_key = exercise_number[0]

        # limpiar el número de ejercicio
        exercise_number = re.sub(r"[\.-]", "", exercise_number[0])

        # ejercicios de formulación - NO CLASIFICAR

        # condiciones a cumplir para ser ejercicio de formulación
        nomenclature_pre_2020 = exam_year <= 2019 and exercise_number_key == "1"
        nomenclature_2020_2023 =  (2020 <= exam_year <= 2023) and exercise_number_key == "A"

        if nomenclature_pre_2020 or nomenclature_2020_2023:
            exercise_topic = "Formulación"
            exercise_types = ["Formulación"]

        elif exam_year >= 2024 and exercise_number_key == "4A":
            exercise_topic = "Formulación"
            exercise_types = ["Formulación orgánica"]

        else:

            ###### CLASIFICACIÓN #######

            # opciones de tema para el ejercicio suministrado
            exercise_topic_options = exercise_topics_dict[exercise_number_key]

            # recorro los posibles temas del ejercicio
            # e intento clasificar el tema del ejercicio
            exercise_topic = ""
            for topic in exercise_topic_options:
                
                # obtengo las palabras clave que identifican el tema
                # y los tipos de ejercicios del tema y sus respectivas keywords
                topic_identifying_keywords = keywords[topic]["keywords_tema"]
                for kw in topic_identifying_keywords:
                    if kw in statement:
                        exercise_topic = topic
                        break

                # si se ha clasificado el ejercicio deja de buscar
                if exercise_topic:
                    break

            # lista de tipos de ejercicio
            exercise_types = []

            # si se ha identificado el tema
            # lo siguiente es asignar tipo(s) de ejercicio
            if exercise_topic:
                # compruebo existencia de keywords de cada tipo de ejercicio en el enunciado
                exercise_type_keywords = keywords[exercise_topic]["tipos"]
                for t, kw_list in exercise_type_keywords.items():
                    # comprueba lista de keywords para ese tipo de ejercicio
                    # con que haga match una keyword con el enunciado
                    # clasifícalo com ese tipo de ejercicio
                    if any(kw in statement for kw in kw_list):
                        exercise_types.append(t)
            else:
                exercise_topic, exercise_types = None, [] 
    else:
        exercise_topic, exercise_types = None, []

    return (exam_year, exam_name, exercise_number, exercise_topic, exercise_types, statement)

In [5]:
# file = ROOT / "examenes_quimica/2013_quimica/2013_quimica_septiembre_reserva_a.pdf"

In [6]:
# exam_text = ext.extract_exercises_from_pdf(file)

# exam_year = int(file.name.split("_")[0])

# if exam_year < 2020:
#     exam_period = "2010-2019"
# elif exam_year <= 2024:
#     exam_period = "2020-2024"
# else:
#     exam_period = "2025+"

# exercises_list = ext.split_exercises(exam_text, exam_period=exam_period)

In [7]:
# print(exercises_list[11][1])

In [8]:
# classify_exercise(file.stem, exercises_list[11][1], quimica_keywords)

In [9]:
classified_exercises = []
for file in (ROOT / "examenes_quimica_renombrados/").rglob("*.pdf"):
    if file.is_file():
        exam_text = ext.extract_exercises_from_pdf(file)
        exam_year = int(file.name.split("_")[0])

        if exam_year < 2020:
            exam_period = "2010-2019"
        elif exam_year <= 2024:
            exam_period = "2020-2024"
        else:
            exam_period = "2025+"

        exercises_tuples = ext.split_exercises(exam_text, exam_period=exam_period)

        # exercises_list devuelve una tupla: (exam_option, statement)
        for exercise_tuple in exercises_tuples:
            try:
                statement = exercise_tuple[1]
                exam_option = exercise_tuple[0]
                classification = classify_exercise(file.name, statement, keywords=quimica_keywords)
                classification = list(classification)
                classification.append(exam_option)
                classified_exercises.append(classification)
            except Exception as e:
                print(f"Error: file {file.name}. Exception: {e}")

df = pd.DataFrame(classified_exercises, columns = ["year", "exam_name", "exercise_number", "topic", "exercise_type", "statement", "exam_option"])
df = df.explode("exercise_type", ignore_index=True)

In [10]:
df

,year,exam_name,exercise_number,topic,exercise_type,statement,exam_option
0,2011,Quimica - 123.pdf,1,Formulación,Formulación,1.- Formule o nombre los compuestos siguientes...,Opción A
1,2011,Quimica - 123.pdf,2,enlace_químico,Propiedades en función del tipo de enlace,"2.- Dadas las especies Cl₂, KCl, Fe y H₂O:\na)...",Opción A
2,2011,Quimica - 123.pdf,2,enlace_químico,Tipo de enlace,"2.- Dadas las especies Cl₂, KCl, Fe y H₂O:\na)...",Opción A
3,2011,Quimica - 123.pdf,3,solubilidad,efecto del ion común - teórico,3.- Se dispone de una disolución acuosa satura...,Opción A
4,2011,Quimica - 123.pdf,4,enlace_químico,NaN,"4.- Dada la reacción 2 H(g) ¾¾→ H₂(g), contest...",Opción A
...,...,...,...,...,...,...,...
1780,2019,Quimica - suplente - septiembre - - a - y - b...,4,NaN,NaN,4.- Complete las siguientes reacciones e indiq...,Opción A
1781,2019,Quimica - suplente - septiembre - - a - y - b...,5,ácido_base,disoluciones,5.- a) El ácido cloroacético� ⎯(⎯C⎯lC⎯⎯H⎯₂⎯C⎯O...,Opción A
1782,2019,Quimica - suplente - septiembre - - a - y - b...,5,ácido_base,neutralización,5.- a) El ácido cloroacético� ⎯(⎯C⎯lC⎯⎯H⎯₂⎯C⎯O...,Opción A
1783,2019,Quimica - suplente - septiembre - - a - y - b...,5,ácido_base,ácido base débil,5.- a) El ácido cloroacético� ⎯(⎯C⎯lC⎯⎯H⎯₂⎯C⎯O...,Opción A


In [11]:
df[df.topic.isna()]

,year,exam_name,exercise_number,topic,exercise_type,statement,exam_option
34,2011,Quimica - 122a.pdf,4,NaN,NaN,4.- Se dispone de 2 litros de disolución acuos...,Opción A
73,2011,Quimica - 121.pdf,5,NaN,NaN,5.- A 25ºC una disolución acuosa de amoniaco c...,Opción A
76,2011,Quimica - 121.pdf,2,NaN,NaN,"2.- a) ¿Cuál es la masa, expresada en gramos, ...",Opción A
79,2011,Quimica - 121.pdf,4,NaN,NaN,4.- Al disolver en agua las siguientes sales: ...,Opción A
80,2011,Quimica - 121.pdf,5,NaN,NaN,5.- La reacción utilizada para la soldadura al...,Opción A
...,...,...,...,...,...,...,...
1512,2014,Quimica - 4 - a - b.pdf,2,NaN,NaN,2.- Justifique la veracidad o falsedad de las ...,Opción A
1517,2014,Quimica - 4 - a - b.pdf,6,NaN,NaN,6.- Dada la siguiente reacción: As + KBrO + KO...,Opción B
1536,2014,Quimica - 2 - a - b.pdf,3,NaN,NaN,3.- Justifique qué ocurrirá cuando:\na) Un cla...,Opción A
1617,2023,Quimica - titular - b.pdf,B6,NaN,NaN,B6. Escriba y ajuste las siguientes reacciones...,NaN


In [12]:
df[df.exercise_type == "Formulación"].shape

(170, 7)

In [13]:
len(list((ROOT / "examenes_quimica_renombrados").rglob("*.pdf")))

109

In [ ]:
nomenclature_classified_df = df.loc[df.exercise_type == "Formulación", ["year", "exam_name", "statement"]]

In [46]:
def extraer_nombrar_compuestos(statement):
    start_position = statement.find("a)")
    end_position = statement.find("d)") - 1
    return statement[start_position:end_position]

In [ ]:
nomenclature_classified_df.statement = nomenclature_classified_df.statement.apply(lambda st: extraer_nombrar_compuestos(st))

In [ ]:
nomenclature_classified_df

,year,exam_name,statement
0,2011,Quimica - 123.pdf,a) Dióxido de azufre b) Nitrato de amonio\nc) ...
7,2011,Quimica - 123.pdf,a) Carbonato de aluminio b) Yoduro de plomo (I...
15,2011,Quimica - 221a.pdf,a) Óxido de calcio b) Ácido bórico\nc) Hepta-2...
22,2011,Quimica - 221a.pdf,a) Sulfuro de manganeso (III) b) Hidrogenocarb...
31,2011,Quimica - 122a.pdf,a) Ácido fosfórico b) Permanganato de bario\nc...
...,...,...,...
1733,2019,Quimica - titular - junio - - a - y - b.pdf,a) Peróxido de estroncio; b) Bromuro de hidróg...
1742,2019,Quimica - reserva - b - - a - y - b.pdf,a) Óxido de cobalto(III); b) Nitrato de hierro...
1752,2019,Quimica - reserva - b - - a - y - b.pdf,a) Sulfuro de cobre(II); b) Ácido carbónico;\n...
1763,2019,Quimica - suplente - septiembre - - a - y - b...,a) Peróxido de sodio; b) Sulfuro de cinc; c) p...


In [ ]:
with open("nomenclature_statements.json", "r") as file:
    nomenclature_statements_dict = json.load(file)

In [ ]:
nomenclature_emestrada_df = pd.DataFrame([
    {"year": int(year), "exam_name": exam_name, "statement": statement}
    for year, exercises in nomenclature_statements_dict.items()      # recorre cada año
    for exam_name, statement in exercises        # recorre cada [titulo, enunciado] de ese año
])

In [ ]:
nomenclature_emestrada_df.statement = nomenclature_emestrada_df.statement.apply(lambda x: x.replace(";", ""))

In [ ]:
nomenclature_emestrada_df

,year,exam_name,statement
0,2008,química. 2008. junio . opción a,a) Permanganato de cobalto(II) b) Ácido bórico...
1,2008,química. 2008. junio . opción b,a) Ácido perclórico b) Seleniuro de hidrógeno\...
2,2008,química. 2008. reserva 1 . opción a,a) Hidrogenocarbonato de sodio. b) Óxido de\no...
3,2008,química. 2008. reserva 1 . opción b,a) Hidróxido de galio b) Bromato de estroncio\...
4,2008,química. 2008. reserva 2 . opción a,a) Dicromato de potasio b) Hidróxido de\nvanad...
...,...,...,...
297,2007,química. 2007. reserva 3. . opción b,a) Permanganato de potasio b) Seleniuro de\npl...
298,2007,química. 2007. reserva 4. . opción a,a) Yodito de estroncio b) Hidróxido de\nestaño...
299,2007,química. 2007. reserva 4. . opción b,a) Cromato de paladio(II) b) Nitrito de\ncobre...
300,2007,química. 2007. septiembre. . opción a,a) Ácido sulfúrico b) Hipoclorito de sodio\nc)...


In [ ]:
nomenclature_classified_df.loc[nomenclature_classified_df.year == 2011]

,year,exam_name,statement
0,2011,Quimica - 123.pdf,a) Dióxido de azufre b) Nitrato de amonio\nc) ...
7,2011,Quimica - 123.pdf,a) Carbonato de aluminio b) Yoduro de plomo (I...
15,2011,Quimica - 221a.pdf,a) Óxido de calcio b) Ácido bórico\nc) Hepta-2...
22,2011,Quimica - 221a.pdf,a) Sulfuro de manganeso (III) b) Hidrogenocarb...
31,2011,Quimica - 122a.pdf,a) Ácido fosfórico b) Permanganato de bario\nc...
37,2011,Quimica - 122a.pdf,a) Peróxido de Bario b) Óxido de cobalto (III)...
46,2011,Quimica - 222.pdf,a) Hidróxido de hierro (II) b) Sulfuro de hidr...
57,2011,Quimica - 222.pdf,a) Peróxido de sodio b) Hidrogenosulfito de ci...
64,2011,Quimica - 121.pdf,a) Óxido de níquel (III) b) Hidróxido de estro...
75,2011,Quimica - 121.pdf,a) Ácido hipobromoso b) Hidróxido de cobre (II...


In [ ]:
nomenclature_emestrada_df.loc[nomenclature_emestrada_df.year == 2011].sort_values("year")

,year,exam_name,statement
242,2011,química. 2011. junio. . opción a,a) Óxido de níquel(III); b) Hidróxido de\nestr...
243,2011,química. 2011. junio. . opción b,a) Ácido hipobromoso; b) Hidróxido de\ncobre(I...
244,2011,química. 2011. reserva 1. . opción a,a) Ácido fosfórico; b) Permanganato de bario;\...
245,2011,química. 2011. reserva 1. . opción b,a) Peróxido de Bario; b) Óxido de cobalto(III)...
246,2011,química. 2011. reserva 2. . opción a,a) Dióxido de azufre; b) Nitrato de amonio;\nc...
247,2011,química. 2011. reserva 2. . opción b,a) Carbonato de aluminio; b) Yoduro de\nplomo(...
248,2011,química. 2011. reserva 3. . opción a,a) Hidróxido de hierro(II); b) Sulfuro de\nhid...
249,2011,química. 2011. reserva 3. . opción b,a) Peróxido de sodio; b) Hidrogenosulfito de\n...
250,2011,química. 2011. reserva 4. . opción a,a) Óxido de níquel(II); b) Carbonato de sodio;...
251,2011,química. 2011. reserva 4. . opción b,a) Bromuro de hidrógeno; b) Ácido nitroso;\nc)...


In [ ]:
pd.merge(nomenclature_classified_df, nomenclature_emestrada_df, on="statement")

,year_x,exam_name_x,statement,year_y,exam_name_y
0,2011,Quimica - 123.pdf,a) Dióxido de azufre b) Nitrato de amonio\nc) ...,2011,química. 2011. reserva 2. . opción a
1,2011,Quimica - 221a.pdf,a) Óxido de calcio b) Ácido bórico\nc) Hepta-2...,2011,química. 2011. septiembre. . opción a
2,2011,Quimica - 122a.pdf,a) Ácido fosfórico b) Permanganato de bario\nc...,2011,química. 2011. reserva 1. . opción a
3,2011,Quimica - 223a.pdf,a) Bromuro de hidrógeno b) Ácido nitroso\nc) 2...,2011,química. 2011. reserva 4. . opción b
4,2016,Quimica - titular - septiembre - 1 - a - b.pdf,a) Óxido de platino(II) b) Sulfito de cadmio\n...,2016,química. 2016. septiembre. . opción a
5,2016,Quimica - titular - septiembre - 1 - a - b.pdf,a) Hidruro de estaño(IV) b) Ácido carbónico\nc...,2016,química. 2016. septiembre. . opción b
6,2015,Quimica - 4.pdf,a) Óxido de manganeso(VII) b) Ácido clórico\nc...,2015,química. 2015. junio. . opción b
7,2015,Quimica - 3.pdf,a) Peróxido de bario b) Sulfuro de galio(III)\...,2015,química. 2015. septiembre. . opción a
8,2015,Quimica - 3.pdf,a) Arseniato de cobalto(II) b) Hidróxido de\nm...,2015,química. 2015. septiembre. . opción b
9,2015,Quimica - 1.pdf,a) Óxido de vanadio(V) b) Nitrato de calcio\nc...,2006,química. 2006. reserva 3. . opción a
